# 20 — Qwen3.8-27B zero-shot CTD sanity check (Biowulf OnDemand)

Biowulf/Open OnDemand version of the strong-model go/no-go experiment. No Colab-specific paths, no Google Drive mount, and no 4-bit quantization. Intended for an A100 80 GB interactive Jupyter session.

Repository root: `/data/LunaLab/yoshi/llm-tuning-playground`

Primary quantities: **Clean**, **Distractor-5** (evidence selection), and **Hard no-path** (evidence sufficiency). One DiseaseID split, one seed, and 50 examples per condition are enough for the first decision.


In [1]:
# Environment and paths — run this first.
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path("/data/LunaLab/yoshi/llm-tuning-playground")
assert ROOT.exists(), f"Repository root not found: {ROOT}"

# Load secrets / environment variables.
ENV_FILE = ROOT / ".env"
load_dotenv(ENV_FILE)

# Persistent Hugging Face cache.
# Respect HF_HOME from .env if provided; otherwise use the repo-local cache.
HF_HOME = Path(
    os.environ.get("HF_HOME", str(ROOT / "hf_cache"))
)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME / "hub")

HF_HOME.mkdir(parents=True, exist_ok=True)

# Project paths.
DATA_DIR = ROOT / "ctd_data"
OUT_DIR = ROOT / "results" / "20_biowulf"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity checks.
print("ROOT:", ROOT)
print("ENV_FILE:", ENV_FILE)
print("HF_HOME:", HF_HOME)
print("OUT_DIR:", OUT_DIR)
print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))

ROOT: /data/LunaLab/yoshi/llm-tuning-playground
ENV_FILE: /data/LunaLab/yoshi/llm-tuning-playground/.env
HF_HOME: /data/LunaLab/yoshi/llm-tuning-playground/hf_cache
OUT_DIR: /data/LunaLab/yoshi/llm-tuning-playground/results/20_biowulf
HF_TOKEN loaded: True


In [15]:
import re, gzip, random, requests
import numpy as np
import pandas as pd
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_NAME = 'Qwen/Qwen3.8-27B'
SPLIT = 'DiseaseID'
SEED = 1
N_EVAL = 50
MAX_NEW_TOKENS = 128

assert torch.cuda.is_available(), 'No CUDA GPU is visible. Launch the OnDemand Jupyter session with an A100 GPU.'
props = torch.cuda.get_device_properties(0)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(props.total_memory / 1024**3, 1))
print('BF16 supported:', torch.cuda.is_bf16_supported())
print('Model:', MODEL_NAME)


GPU: NVIDIA A100-SXM4-80GB
GPU memory (GiB): 79.2
BF16 supported: True
Model: Qwen/Qwen3.8-27B


In [9]:
# CTD acquisition / parsing. Existing files under ROOT/ctd_data are reused.
CHEM_NAME = 'CTD_chem_gene_ixns.tsv.gz'
GD_NAMES = ['CTD_curated_genes_diseases.tsv.gz', 'CTD_genes_diseases.tsv.gz']

def valid_gzip(path, min_bytes=10000):
    path = Path(path)
    if not path.exists() or path.stat().st_size < min_bytes:
        return False
    try:
        with open(path, 'rb') as f:
            if f.read(2) != b'\x1f\x8b':
                return False
        with gzip.open(path, 'rb') as f:
            f.read(128)
        return True
    except Exception:
        return False

def ensure_file(names):
    if isinstance(names, str):
        names = [names]
    for name in names:
        p = DATA_DIR / name
        if valid_gzip(p):
            print('Found:', p)
            return p
    for name in names:
        dest = DATA_DIR / name
        for url in [f'https://ctdbase.org/reports/{name}', f'https://ctdbase.org/downloads/{name}']:
            try:
                print('Downloading:', url)
                with requests.get(url, stream=True, timeout=(20, 300), headers={'User-Agent':'Mozilla/5.0'}) as r:
                    r.raise_for_status()
                    with open(dest, 'wb') as f:
                        for chunk in r.iter_content(1024 * 1024):
                            if chunk:
                                f.write(chunk)
                if valid_gzip(dest):
                    return dest
            except Exception as e:
                print('failed:', type(e).__name__, str(e)[:120])
            dest.unlink(missing_ok=True)
    raise FileNotFoundError(names)

def read_ctd(path, expected):
    header = None
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if not line.startswith('#'):
                break
            s = line.lstrip('#').strip()
            if '\t' in s:
                cols = [x.strip() for x in s.split('\t')]
                if any(x in cols for x in expected):
                    header = cols
    if header is None:
        raise ValueError(f'Header not found: {path}')
    return pd.read_csv(path, sep='\t', comment='#', compression='gzip', dtype=str,
                       low_memory=False, header=None, names=header)

cg = read_ctd(ensure_file(CHEM_NAME), ['ChemicalName','ChemicalID','GeneSymbol','GeneID'])
gd = read_ctd(ensure_file(GD_NAMES), ['GeneSymbol','GeneID','DiseaseName','DiseaseID'])

cg2 = cg[['ChemicalName','ChemicalID','GeneSymbol','GeneID']].dropna().drop_duplicates()
gd2 = gd[['GeneSymbol','GeneID','DiseaseName','DiseaseID']].dropna().drop_duplicates()
paths = cg2.merge(gd2, on=['GeneSymbol','GeneID']).drop_duplicates()
paths = paths[(paths.ChemicalName.str.len() < 100) & (paths.DiseaseName.str.len() < 120)].reset_index(drop=True)
edge_pool = gd2[['GeneSymbol','DiseaseName']].drop_duplicates().reset_index(drop=True)

print('paths:', len(paths), 'edges:', len(edge_pool))


Found: /data/LunaLab/yoshi/llm-tuning-playground/ctd_data/CTD_chem_gene_ixns.tsv.gz
Found: /data/LunaLab/yoshi/llm-tuning-playground/ctd_data/CTD_curated_genes_diseases.tsv.gz
paths: 9707313 edges: 34272


In [10]:
# One entity-disjoint test split; only the 3 conditions needed for the go/no-go check.
rng_np = np.random.default_rng(SEED)
entities = paths[SPLIT].dropna().unique().copy()
rng_np.shuffle(entities)
cut = max(1, int(0.8 * len(entities)))
test_entities = set(entities[cut:])
test_pool = paths[paths[SPLIT].isin(test_entities)].drop_duplicates(['ChemicalID','GeneID','DiseaseID'])
assert len(test_pool) >= N_EVAL
test = test_pool.sample(N_EVAL, random_state=1000 + SEED).reset_index(drop=True)

def render(row, edges):
    ev = '\n'.join(f'- {g} -> {d}' for g, d in edges)
    return (
        'Use only the supplied evidence. Determine the disease supported by the path from the queried chemical through the queried gene. '
        'If no supplied gene-disease relation supports the queried gene, answer exactly: No supported path.\n\n'
        f'Chemical: {row.ChemicalName}\n'
        f'Gene: {row.GeneSymbol}\n'
        f'Evidence:\n{ev}'
    )

def positive_edges(row, k, rng):
    out = [(str(row.GeneSymbol), str(row.DiseaseName))]
    if k > 0:
        pool = edge_pool[(edge_pool.GeneSymbol != row.GeneSymbol) &
                         (edge_pool.DiseaseName != row.DiseaseName)]
        sub = pool.sample(k, random_state=rng.randint(0, 2**31 - 1))
        out += [(str(g), str(d)) for g, d in sub.itertuples(index=False, name=None)]
    rng.shuffle(out)
    return out

def no_path_edges(row, k, rng):
    pool = edge_pool[(edge_pool.GeneSymbol != row.GeneSymbol) &
                     (edge_pool.DiseaseName != row.DiseaseName)]
    sub = pool.sample(k, random_state=rng.randint(0, 2**31 - 1))
    out = [(str(g), str(d)) for g, d in sub.itertuples(index=False, name=None)]
    rng.shuffle(out)
    return out

rng = random.Random(20000 + SEED)
sets = {'clean': [], 'distractor_5': [], 'hard_no_path': []}
for _, row in test.iterrows():
    sets['clean'].append({'prompt': render(row, positive_edges(row, 0, rng)),
                          'target': str(row.DiseaseName), 'no_path': False})
    sets['distractor_5'].append({'prompt': render(row, positive_edges(row, 5, rng)),
                                 'target': str(row.DiseaseName), 'no_path': False})
    sets['hard_no_path'].append({'prompt': render(row, no_path_edges(row, 5, rng)),
                                 'target': None, 'no_path': True})

print({k: len(v) for k, v in sets.items()})


{'clean': 50, 'distractor_5': 50, 'hard_no_path': 50}


In [11]:
# Load the model in BF16. This is preferable to 4-bit for the strong-model sanity check.
# A100 80 GB should have enough memory for a 27B BF16 model plus short generation.
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()

print('loaded')
print('allocated GiB:', round(torch.cuda.memory_allocated() / 1024**3, 2))
print('reserved GiB:', round(torch.cuda.memory_reserved() / 1024**3, 2))


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


loaded
allocated GiB: 22.39
reserved GiB: 73.36


In [12]:
def generate_one(prompt):
    messages = [{'role':'user', 'content':[{'type':'text', 'text':prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    dev = next(model.parameters()).device
    inputs = {k: v.to(dev) if hasattr(v, 'to') else v for k, v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    n = inputs['input_ids'].shape[-1]
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

def norm(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())

def score(item, pred):
    p = norm(pred)
    if item['no_path']:
        return int('no supported path' in p)
    return int(norm(item['target']) in p and 'no supported path' not in p)

RESULT_CSV = OUT_DIR / '20_qwen38_27b_results.csv'
SUMMARY_CSV = OUT_DIR / '20_qwen38_27b_summary.csv'

rows = []
for condition, items in sets.items():
    print('\n' + '=' * 70)
    print(condition)
    print('=' * 70)
    for i, item in enumerate(items):
        pred = generate_one(item['prompt'])
        ok = score(item, pred)
        rows.append({
            'model': MODEL_NAME, 'split': SPLIT, 'seed': SEED,
            'condition': condition, 'i': i, 'target': item['target'],
            'prediction': pred, 'correct': ok
        })
        if i < 3:
            print(i, ok, pred[:180])
        if (i + 1) % 10 == 0:
            pd.DataFrame(rows).to_csv(RESULT_CSV, index=False)
            print(f'checkpoint {i + 1}/{len(items)}')
    acc = np.mean([r['correct'] for r in rows if r['condition'] == condition])
    print('accuracy:', round(float(acc), 3))

df = pd.DataFrame(rows)
df.to_csv(RESULT_CSV, index=False)
summary = df.groupby('condition', as_index=False).correct.mean().rename(columns={'correct':'accuracy'})
summary.to_csv(SUMMARY_CSV, index=False)
display(summary)
print('Saved:', RESULT_CSV)



clean
0 1 Drug Hypersensitivity
1 1 Asthma
2 1 Cerebral Hemorrhage
checkpoint 10/50
checkpoint 20/50
checkpoint 30/50
checkpoint 40/50
checkpoint 50/50
accuracy: 0.94

distractor_5
0 1 Drug Hypersensitivity
1 0 No supported path
2 1 Cerebral Hemorrhage
checkpoint 10/50
checkpoint 20/50
checkpoint 30/50
checkpoint 40/50
checkpoint 50/50
accuracy: 0.9

hard_no_path
0 1 No supported path
1 1 No supported path
2 1 No supported path
checkpoint 10/50
checkpoint 20/50
checkpoint 30/50
checkpoint 40/50
checkpoint 50/50
accuracy: 1.0


,condition,accuracy
0,clean,0.94
1,distractor_5,0.90
2,hard_no_path,1.00


Saved: /data/LunaLab/yoshi/llm-tuning-playground/results/20_biowulf/20_qwen38_27b_results.csv


In [13]:
# Descriptive go/no-go rule — not a statistical test.
m = dict(zip(summary.condition, summary.accuracy))
clean = m['clean']
d5 = m['distractor_5']
hard = m['hard_no_path']
gap = d5 - hard

print(f'Clean={clean:.3f} | D5={d5:.3f} | Hard NP={hard:.3f} | D5-Hard gap={gap:+.3f}')
if clean >= 0.95 and d5 >= 0.95 and hard >= 0.95:
    print('GO/NO-GO: near ceiling on the strong model -> broad reliability claim is weak for this simple task.')
elif d5 >= 0.90 and hard <= 0.80:
    print('GO/NO-GO: clear selection-sufficiency dissociation persists -> worth continuing.')
else:
    print('GO/NO-GO: intermediate result -> inspect predictions and repeat with a second model/family before deciding.')


Clean=0.940 | D5=0.900 | Hard NP=1.000 | D5-Hard gap=-0.100
GO/NO-GO: intermediate result -> inspect predictions and repeat with a second model/family before deciding.


In [16]:
# ============================================================
# Strong-model extension: run all three entity-disjoint splits
# ============================================================

SPLITS = ["ChemicalID", "GeneID", "DiseaseID"]
N_EVAL_PER_SPLIT = 50

ALL_RESULT_CSV = OUT_DIR / "20_qwen38_27b_all_splits_results.csv"
ALL_SUMMARY_CSV = OUT_DIR / "20_qwen38_27b_all_splits_summary.csv"


def build_eval_sets(split, seed=1, n_eval=50):
    """Build matched Clean / Distractor-5 / Hard-no-path sets for one entity-disjoint split."""

    rng_np = np.random.default_rng(seed)

    entities = paths[split].dropna().unique().copy()
    rng_np.shuffle(entities)

    cut = max(1, int(0.8 * len(entities)))
    test_entities = set(entities[cut:])

    test_pool = (
        paths[paths[split].isin(test_entities)]
        .drop_duplicates(["ChemicalID", "GeneID", "DiseaseID"])
    )

    assert len(test_pool) >= n_eval, (
        f"Not enough examples for {split}: "
        f"{len(test_pool)} available, {n_eval} requested"
    )

    test = (
        test_pool
        .sample(n_eval, random_state=1000 + seed)
        .reset_index(drop=True)
    )

    rng = random.Random(20000 + seed)

    eval_sets = {
        "clean": []def build_nested_distractors(row, max_k=50, seed=1, example_idx=0):
    """
    Sample distractors once.
    D1 is a prefix of D3, D3 is a prefix of D5, etc.
    """
    pool = edge_pool[
        (edge_pool.GeneSymbol != row.GeneSymbol) &
        (edge_pool.DiseaseName != row.DiseaseName)
    ].drop_duplicates()

    assert len(pool) >= max_k

    rs = 50000 + seed * 1000 + example_idx

    sub = pool.sample(
        max_k,
        random_state=rs,
        replace=False,
    )

    distractors = [
        (str(g), str(d))
        for g, d in sub.itertuples(index=False, name=None)
    ]

    return distractors
        "distractor_5": [],
        "hard_no_path": [],
    }

    for _, row in test.iterrows():
        eval_sets["clean"].append({
            "prompt": render(row, positive_edges(row, 0, rng)),
            "target": str(row.DiseaseName),
            "no_path": False,
        })

        eval_sets["distractor_5"].append({
            "prompt": render(row, positive_edges(row, 5, rng)),
            "target": str(row.DiseaseName),
            "no_path": False,
        })

        eval_sets["hard_no_path"].append({
            "prompt": render(row, no_path_edges(row, 5, rng)),
            "target": None,
            "no_path": True,
        })

    return eval_sets


all_rows = []

for split in SPLITS:
    print("\n" + "#" * 80)
    print(f"SPLIT: {split}")
    print("#" * 80)

    split_sets = build_eval_sets(
        split=split,
        seed=SEED,
        n_eval=N_EVAL_PER_SPLIT,
    )

    for condition, items in split_sets.items():
        print("\n" + "=" * 70)
        print(f"{split} | {condition}")
        print("=" * 70)

        condition_rows = []

        for i, item in enumerate(items):
            pred = generate_one(item["prompt"])
            ok = score(item, pred)

            row = {
                "model": MODEL_NAME,
                "split": split,
                "seed": SEED,
                "condition": condition,
                "i": i,
                "target": item["target"],
                "prediction": pred,
                "correct": ok,
            }

            all_rows.append(row)
            condition_rows.append(row)

            if i < 3:
                print(i, ok, pred[:180])

            if (i + 1) % 10 == 0:
                pd.DataFrame(all_rows).to_csv(
                    ALL_RESULT_CSV,
                    index=False,
                )
                print(f"checkpoint {i + 1}/{len(items)}")

        acc = np.mean([r["correct"] for r in condition_rows])

        print("accuracy:", round(float(acc), 3))


# Save full predictions.
all_df = pd.DataFrame(all_rows)
all_df.to_csv(ALL_RESULT_CSV, index=False)

# Per-split summary.
all_summary = (
    all_df
    .groupby(["split", "condition"], as_index=False)
    .correct
    .mean()
    .rename(columns={"correct": "accuracy"})
)

all_summary.to_csv(ALL_SUMMARY_CSV, index=False)

display(all_summary)

print("Saved:", ALL_RESULT_CSV)
print("Saved:", ALL_SUMMARY_CSV)


################################################################################
SPLIT: ChemicalID
################################################################################

ChemicalID | clean
0 1 Macrocephaly, Alopecia, Cutis Laxa, and Scoliosis
1 1 Prostatic Neoplasms
2 1 Heart Failure
checkpoint 10/50
checkpoint 20/50
checkpoint 30/50
checkpoint 40/50
checkpoint 50/50
accuracy: 0.94

ChemicalID | distractor_5
0 1 Macrocephaly, Alopecia, Cutis Laxa, and Scoliosis
1 0 No supported path
2 0 No supported path
checkpoint 10/50
checkpoint 20/50
checkpoint 30/50
checkpoint 40/50
checkpoint 50/50
accuracy: 0.8

ChemicalID | hard_no_path
0 1 No supported path
1 1 No supported path
2 1 No supported path
checkpoint 10/50
checkpoint 20/50
checkpoint 30/50
checkpoint 40/50
checkpoint 50/50
accuracy: 1.0

################################################################################
SPLIT: GeneID
################################################################################

GeneID | 

KeyboardInterrupt: 

In [ ]:
# Compact split × condition table.

pivot = (
    all_summary
    .pivot(
        index="split",
        columns="condition",
        values="accuracy",
    )
    .reindex(columns=["clean", "distractor_5", "hard_no_path"])
)

display(pivot.round(3))

In [19]:
# Inspect false abstentions:
# supported examples where the model incorrectly says "No supported path".

false_abstentions = all_df[
    (all_df["condition"].isin(["clean", "distractor_5"]))
    & (all_df["correct"] == 0)
    & (
        all_df["prediction"]
        .str.lower()
        .str.contains("no supported path", na=False)
    )
].copy()

display(
    false_abstentions[
        ["split", "condition", "i", "target", "prediction"]
    ]
)

print("False abstentions:", len(false_abstentions))

NameError: name 'all_df' is not defined

In [17]:
# ============================================================
# Distractor scaling experiment
# Same underlying examples + nested distractors across k
# ============================================================

K_VALUES = [0, 1, 3, 5, 10, 20, 50]
SPLITS = ["ChemicalID", "GeneID", "DiseaseID"]
N_SCALING = 50

SCALING_RESULTS_CSV = OUT_DIR / "21_qwen38_27b_distractor_scaling_results.csv"
SCALING_SUMMARY_CSV = OUT_DIR / "21_qwen38_27b_distractor_scaling_summary.csv"

In [18]:
def get_fixed_test_examples(split, seed=1, n_eval=50):
    """
    Same entity-disjoint split logic as before.
    Returns a fixed set of underlying positive examples.
    """
    rng_np = np.random.default_rng(seed)

    entities = paths[split].dropna().unique().copy()
    rng_np.shuffle(entities)

    cut = max(1, int(0.8 * len(entities)))
    test_entities = set(entities[cut:])

    test_pool = (
        paths[paths[split].isin(test_entities)]
        .drop_duplicates(["ChemicalID", "GeneID", "DiseaseID"])
    )

    assert len(test_pool) >= n_eval

    test = (
        test_pool
        .sample(n_eval, random_state=1000 + seed)
        .reset_index(drop=True)
    )

    return test

In [20]:
def build_nested_distractors(row, max_k=50, seed=1, example_idx=0):
    """
    Sample distractors once.
    D1 is a prefix of D3, D3 is a prefix of D5, etc.
    """
    pool = edge_pool[
        (edge_pool.GeneSymbol != row.GeneSymbol) &
        (edge_pool.DiseaseName != row.DiseaseName)
    ].drop_duplicates()

    assert len(pool) >= max_k

    rs = 50000 + seed * 1000 + example_idx

    sub = pool.sample(
        max_k,
        random_state=rs,
        replace=False,
    )

    distractors = [
        (str(g), str(d))
        for g, d in sub.itertuples(index=False, name=None)
    ]

    return distractors

In [21]:
def make_scaling_item(row, distractors, k, seed, example_idx):
    true_edge = (str(row.GeneSymbol), str(row.DiseaseName))

    edges = [true_edge] + distractors[:k]

    # Deterministic shuffle for reproducibility.
    rng = random.Random(
        70000
        + seed * 10000
        + example_idx * 100
        + k
    )
    rng.shuffle(edges)

    true_edge_position = edges.index(true_edge)

    return {
        "prompt": render(row, edges),
        "target": str(row.DiseaseName),
        "no_path": False,
        "edges": edges,
        "true_edge_position": true_edge_position,
        "n_edges": len(edges),
    }

In [22]:
def classify_positive_error(item, prediction):
    pred = prediction.strip()
    target = str(item["target"]).strip()

    if pred.lower() == target.lower():
        return "correct"

    if "no supported path" in pred.lower():
        return "false_abstention"

    # Did the model output one of the distractor diseases?
    distractor_diseases = {
        str(d)
        for g, d in item["edges"]
        if str(d).lower() != target.lower()
    }

    if any(
        pred.lower() == d.lower()
        for d in distractor_diseases
    ):
        return "distractor_answer"

    return "other_wrong_answer"

In [ ]:
scaling_rows = []

for split in SPLITS:

    print("\n" + "#" * 80)
    print("SPLIT:", split)
    print("#" * 80)

    test = get_fixed_test_examples(
        split=split,
        seed=SEED,
        n_eval=N_SCALING,
    )

    # Pre-build distractor pools once per underlying example.
    distractor_bank = {}

    for i, row in test.iterrows():
        distractor_bank[i] = build_nested_distractors(
            row=row,
            max_k=max(K_VALUES),
            seed=SEED,
            example_idx=i,
        )

    for k in K_VALUES:

        print("\n" + "=" * 70)
        print(f"{split} | distractor_{k}")
        print("=" * 70)

        k_rows = []

        for i, row in test.iterrows():

            item = make_scaling_item(
                row=row,
                distractors=distractor_bank[i],
                k=k,
                seed=SEED,
                example_idx=i,
            )

            pred = generate_one(item["prompt"])

            error_type = classify_positive_error(
                item,
                pred,
            )

            correct = int(error_type == "correct")

            result = {
                "model": MODEL_NAME,
                "split": split,
                "seed": SEED,
                "k": k,
                "example_idx": i,

                "ChemicalID": row.ChemicalID,
                "GeneID": row.GeneID,
                "DiseaseID": row.DiseaseID,

                "ChemicalName": row.ChemicalName,
                "GeneSymbol": row.GeneSymbol,
                "target": row.DiseaseName,

                "prediction": pred,
                "correct": correct,
                "error_type": error_type,

                "true_edge_position": item["true_edge_position"],
                "n_edges": item["n_edges"],
            }

            scaling_rows.append(result)
            k_rows.append(result)

            if i < 3:
                print(
                    i,
                    correct,
                    error_type,
                    "| pos:",
                    item["true_edge_position"],
                    "|",
                    pred[:120],
                )

            if (i + 1) % 10 == 0:
                pd.DataFrame(scaling_rows).to_csv(
                    SCALING_RESULTS_CSV,
                    index=False,
                )

                print(
                    f"checkpoint {i + 1}/{len(test)}"
                )

        acc = np.mean(
            [r["correct"] for r in k_rows]
        )

        false_abstain_rate = np.mean([
            r["error_type"] == "false_abstention"
            for r in k_rows
        ])

        print(
            f"k={k} | "
            f"accuracy={acc:.3f} | "
            f"false_abstention={false_abstain_rate:.3f}"
        )


scaling_df = pd.DataFrame(scaling_rows)
scaling_df.to_csv(
    SCALING_RESULTS_CSV,
    index=False,
)

print("Saved:", SCALING_RESULTS_CSV)


################################################################################
SPLIT: ChemicalID
################################################################################

ChemicalID | distractor_0
0 1 correct | pos: 0 | Macrocephaly, Alopecia, Cutis Laxa, and Scoliosis
1 1 correct | pos: 0 | Prostatic Neoplasms
2 1 correct | pos: 0 | Heart Failure
checkpoint 10/50


In [ ]:
scaling_summary = (
    scaling_df
    .groupby(["split", "k"], as_index=False)
    .agg(
        accuracy=("correct", "mean"),
        n=("correct", "size"),
        false_abstention_rate=(
            "error_type",
            lambda x: np.mean(x == "false_abstention"),
        ),
        distractor_answer_rate=(
            "error_type",
            lambda x: np.mean(x == "distractor_answer"),
        ),
        other_wrong_rate=(
            "error_type",
            lambda x: np.mean(x == "other_wrong_answer"),
        ),
    )
)

scaling_summary.to_csv(
    SCALING_SUMMARY_CSV,
    index=False,
)

display(
    scaling_summary.round(3)
)

print("Saved:", SCALING_SUMMARY_CSV)